In [1]:
# imports needed for transcriptions:

import whisper
from whisper.utils import get_writer
import yt_dlp
import ffmpeg
from datetime import datetime
import tqdm
import sys
import pysrt
import re
import gc
import torch

c:\Users\melis\.conda\envs\loqio-env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


### 1. Grab audio file from youtube video

In [2]:
# grabbing audio file from youtube video and saving as m4a
def get_yt_audio(yt_url):
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': '%(id)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'm4a',
        }]
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        error_code = ydl.download([yt_url])

    print(error_code)

# grabbing subtitles only if youtube video has them and saving as srt
def get_yt_subtitles(yt_url):
    ydl_opts = {
        'writesubtitles': True,
        # 'writeautomaticsub': True,
        'subtitlesformat': 'srt',
        'skip_download': True,
        'outtmpl': '%(id)s.%(ext)s',
        'no_cache': True,
        # 'subtitleslangs': ['en','zh-Hans'],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        error_code = ydl.download([yt_url])
    
    print(error_code)

### 2. Transcribe audio file using Whisper and add audio timestamps

In [3]:
# Define a custom tqdm class for creating transcription progress bar
class _CustomProgressBar(tqdm.tqdm):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._current = self.n

    def update(self, n):
        super().update(n)
        self._current += n
        # Add your custom progress handling here (e.g., updating a GUI)
        # print(f"Progress: {self._current}/{self.total}") # Optional: for console output

def extract_yt_video_id(yt_url):
    # Extract the video ID from the YouTube URL
    if "v=" in yt_url:
        return yt_url.split("v=")[-1].split("&")[0]
    elif "youtu.be/" in yt_url:
        return yt_url.split("youtu.be/")[-1].split("?")[0]
    else:
        raise ValueError("Invalid YouTube URL")
    
def transcribe_audio(file, language):
    transcribe_module = sys.modules['whisper.transcribe']
    transcribe_module.tqdm.tqdm = _CustomProgressBar

    model = whisper.load_model("large-v2")
    decode_options = {
    "language": language,
    "verbose": False, # verbose=False is often required for the progress bar to appear/callback to work
    }
    model_opt = dict(task="transcribe", **decode_options)
    result = model.transcribe(file, **model_opt)
    writer = get_writer("srt", ".")
    yt_video_id = file.split(".")[0]  # Assuming the file is named as "{video_id}.m4a"
    writer(result, f"{yt_video_id}.srt")
    for segment in result['segments']:
        print(f"[{segment['start']:.2f}s -> {segment['end']:.2f}s] {segment['text']}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        

In [4]:
get_yt_audio("https://www.youtube.com/watch?v=Taw2tAMGIVM")


[youtube] Extracting URL: https://www.youtube.com/watch?v=Taw2tAMGIVM
[youtube] Taw2tAMGIVM: Downloading webpage


[youtube] Taw2tAMGIVM: Downloading android vr player API JSON
[info] Taw2tAMGIVM: Downloading 1 format(s): 140
[download] Destination: Taw2tAMGIVM.m4a
[download] 100% of  101.13MiB in 00:00:03 at 32.49MiB/s    
[FixupM4a] Correcting container of "Taw2tAMGIVM.m4a"
[ExtractAudio] Not converting audio Taw2tAMGIVM.m4a; file is already in target format m4a
0


In [12]:
# get_yt_audio("https://www.youtube.com/watch?v=Taw2tAMGIVM")
# get_yt_audio("https://www.youtube.com/watch?v=oLHfnK9IVZs")
# get_yt_subtitles("https://www.youtube.com/watch?v=oLHfnK9IVZs")
transcribe_audio("Taw2tAMGIVM.m4a", "zh")

100%|█████████▉| 652223/655223 [3:04:40<00:50, 58.86frames/s]  

[0.00s -> 3.00s] 念念我
[3.00s -> 8.00s] 詞曲 李宗盛
[30.00s -> 32.00s] 早安
[32.00s -> 34.00s] 早安
[42.00s -> 44.00s] 美貌check
[46.00s -> 48.00s] check
[48.00s -> 50.00s] 性面的这是
[50.00s -> 52.00s] 我今天幸亏没穿裙子
[52.00s -> 54.00s] 还好没穿小校服
[56.00s -> 58.00s] 欢迎你们来到东巡酒店的天空之镜观景台
[60.00s -> 64.00s] 你们身后就是威尔耸利的慕斯塔格峰
[67.00s -> 68.00s] 我们也是从这个方向来的
[68.00s -> 69.00s] 是的
[69.00s -> 70.00s] 对吧
[70.00s -> 73.00s] 然后在大家右手边是新都库什山脉
[75.00s -> 80.00s] 因此在塔线也能最直观的感受到帕米尔高原山脉的延绵交汇
[82.00s -> 84.00s] 在开始我们今天的任务之前
[84.00s -> 87.00s] 志胜你昨天是怎么回事
[88.00s -> 89.00s] 对啊
[89.00s -> 90.00s] 我也想是
[90.00s -> 93.00s] 我也没想到人在新疆后院失火了
[94.00s -> 95.00s] 说我家里乱
[95.00s -> 96.00s] 我能在实质上当朋友
[96.00s -> 97.00s] 我肯定不是杰皮
[97.00s -> 98.00s] 太吓人了
[98.00s -> 100.00s] 我都不敢往沙发上坐
[101.00s -> 102.00s] 我披个腰
[102.00s -> 103.00s] 对
[103.00s -> 104.00s] 洗脸
[104.00s -> 106.00s] 洗脸也洗手
[106.00s -> 107.00s] 洗脚
[107.00s -> 108.00s] 洗衣服
[108.00s -> 109.00s] 天天洗
[110.00s -> 112.00s] 衣服是洗过的
[112.00s -> 113.00s] 凯哥
[113.00s -> 114.00s] 看完这条热搜
[114.00s -> 115.0

In [35]:
get_yt_audio("https://www.youtube.com/watch?v=oLHfnK9IVZs")
get_yt_subtitles("https://www.youtube.com/watch?v=oLHfnK9IVZs")
transcribe_audio("oLHfnK9IVZs.m4a", "zh")

[youtube] Extracting URL: https://www.youtube.com/watch?v=oLHfnK9IVZs
[youtube] oLHfnK9IVZs: Downloading webpage


[youtube] oLHfnK9IVZs: Downloading android vr player API JSON
[info] oLHfnK9IVZs: Downloading 1 format(s): 140
[download] Destination: oLHfnK9IVZs.m4a
[download] 100% of   14.80MiB in 00:00:00 at 15.45MiB/s    
[FixupM4a] Correcting container of "oLHfnK9IVZs.m4a"
[ExtractAudio] Not converting audio oLHfnK9IVZs.m4a; file is already in target format m4a
0
[youtube] Extracting URL: https://www.youtube.com/watch?v=oLHfnK9IVZs
[youtube] oLHfnK9IVZs: Downloading webpage


[youtube] oLHfnK9IVZs: Downloading android vr player API JSON
[info] oLHfnK9IVZs: Downloading subtitles: zh-Hans
[info] oLHfnK9IVZs: Downloading 1 format(s): 248+251
[info] Writing video subtitles to: oLHfnK9IVZs.zh-Hans.srt


[download] Destination: oLHfnK9IVZs.zh-Hans.srt
[download] 100% of    8.60KiB in 00:00:00 at 71.67KiB/s
0


C:\Users\melis\AppData\Roaming\Python\Python313\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[0.00s -> 22.00s] 【 passed out for 2 days!】
[22.00s -> 27.12s] 早上好,我是家家。
[27.12s -> 34.32s] 我刚刚送了我的女儿去托尔所,
[34.32s -> 39.12s] 然后回家画了一个装。
[39.12s -> 44.00s] 现在我准备做早餐。
[44.00s -> 50.44s] 今天我打算做我最喜欢的早餐。
[50.44s -> 62.20s] 需要的食材很简单,牛油果、吐司、鸡蛋和黄油。
[62.20s -> 66.84s] 那我们先烤面包吧。
[69.84s -> 73.00s] 现在打一个鸡蛋。
[80.44s -> 84.44s] 放一点黄油。
[84.44s -> 96.44s] 黄油融快以后,把蛋倒进去。
[96.44s -> 100.44s] 然后搅拌一下。
[100.44s -> 104.44s] 接下来就很简单。
[104.44s -> 108.44s] 把牛油果抹上去。
[108.44s -> 114.44s] 牛油果和好的,可以用来水培。
[134.44s -> 144.44s] 然后再把鸡蛋放上去。
[144.44s -> 152.44s] 放一点黑胡椒。
[152.44s -> 158.44s] 放一点盐。
[159.44s -> 166.44s] 然后我个人还喜欢放一点拉茶净。
[166.44s -> 168.44s] 因为我喜欢吃辣。
[168.44s -> 172.44s] 放盘子里边。
[172.44s -> 176.44s] 现在做一杯咖啡。
[176.44s -> 182.44s] 这种咖啡糊大家都知道吧。
[183.44s -> 189.44s] 煮咖啡也很简单,只有几个步驟。
[189.44s -> 197.44s] 第一,倒水。
[197.44s -> 207.44s] 第二,放咖啡粉。
[207.44s -> 209.44s] 抖一下。
[209.44s -> 214.44s] 第三,放进去。
[214.44s -> 220.44s] 然后关上。
[220.44s -> 225.44s] 开火。
[227.44s -> 231.44s] 等个一两分钟。
[238.44s -> 244.44s] 好了。
[244.44s -> 250

### 3. Match subtitles to the video timestamps

In [4]:
# 3. turn transcription into dict

def parse_transcription_srt(srt):
    with open(srt, "r", encoding="utf-8") as f:
        srt_content = f.read()

    content = srt_content.replace('\r\n', '\n').strip()
    blocks = re.split(r'\n\n+', content)
    transcript = []

    for block in blocks:
        lines = block.split('\n')
        if (len(lines) >= 3):
            index = lines[0].strip()

            times = lines[1].split(' --> ')

            if (len(times) == 2):
                text = " ".join(lines[2:]).strip()
                transcript.append({
                    "index": index,
                    "start_time": times[0].strip().replace(',', '.'),
                    "end_time": times[1].strip().replace(',', '.'),
                    "text": text
                })

    return transcript

In [42]:
# transcript = parse_transcription_srt("audio.srt")
# print(transcript)

### 4. Put steps 1-3 together to one function

In [5]:
# 4. put all together for api

def transcribe_video(yt_url, language):
    get_yt_audio(yt_url)
    video_id = extract_yt_video_id(yt_url)
    audio_file = f"{video_id}.m4a"
    transcribe_audio(audio_file, language)
    srt_file = f"{video_id}.srt"
    transcript = parse_transcription_srt(srt_file)
    return transcript



In [6]:
transcribe_video("https://www.youtube.com/watch?v=oIoy02pDgoU", "zh")

[youtube] Extracting URL: https://www.youtube.com/watch?v=oIoy02pDgoU
[youtube] oIoy02pDgoU: Downloading webpage


[youtube] oIoy02pDgoU: Downloading android vr player API JSON
[info] oIoy02pDgoU: Downloading 1 format(s): 140
[download] oIoy02pDgoU.m4a has already been downloaded
[download] 100% of   42.79MiB
[ExtractAudio] Not converting audio oIoy02pDgoU.m4a; file is already in target format m4a
0


c:\Users\melis\.conda\envs\loqio-env\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
 91%|█████████▏| 253374/277459 [45:09<04:17, 93.53frames/s] 


[0.00s -> 11.00s] 优优独播剧场——YoYo Television Series Exclusive
[30.00s -> 31.48s] 还是命中注定
[32.16s -> 33.08s] 敬请期待
[33.32s -> 34.24s] 传说曲
[34.56s -> 36.12s] 恶魔宠妃
[38.24s -> 39.16s] 都看完了吗
[39.36s -> 40.04s] 说说吧
[40.32s -> 41.40s] 大少客观啊
[50.32s -> 50.92s] 那个谁
[51.52s -> 52.20s] 你说说
[57.32s -> 58.68s] 这十个项目里面
[60.00s -> 61.12s] 有四个重生
[61.56s -> 62.40s] 三个穿越
[62.68s -> 63.84s] 剩下两个开系统
[64.36s -> 66.24s] 我觉得有点千篇一律
[66.60s -> 67.60s] 这不才九个吗
[68.20s -> 68.84s] 还有一个呢
[69.16s -> 70.20s] 这个传书本啊
[70.72s -> 72.32s] 又老套又过时
[72.60s -> 74.48s] 看紧接就让人脚趾抠地
[78.16s -> 80.20s] 但是老板已经把本全买下了
[80.80s -> 81.64s] 画了这个书
[83.00s -> 84.40s] 新穎 有趣
[84.60s -> 86.20s] 就像一块待爆的葡萄
[87.12s -> 88.88s] 生机勃勃的长头草你是
[89.48s -> 90.44s] 好 既然这样
[90.68s -> 92.44s] 那这块葡萄就交给你来打磨
[92.88s -> 94.00s] 对了 抓紧看完
[94.20s -> 94.96s] 时间不多
[95.16s -> 96.52s] 明天早上我要看到一个
[96.52s -> 98.12s] 五语伦比的改编方向
[98.44s -> 99.28s] 和评估报告
[100.96s -> 101.60s] 要新颖
[102.32s -> 102.96s] 特别
[103.64s -> 104.56s] 明天早上
[113.92s -> 114.36s] 走了啊
[114.36s -> 114.96s] 明儿见
[

[{'index': '1',
  'start_time': '00:00:00.000',
  'end_time': '00:00:11.000',
  'text': '优优独播剧场——YoYo Television Series Exclusive'},
 {'index': '2',
  'start_time': '00:00:30.000',
  'end_time': '00:00:31.480',
  'text': '还是命中注定'},
 {'index': '3',
  'start_time': '00:00:32.160',
  'end_time': '00:00:33.080',
  'text': '敬请期待'},
 {'index': '4',
  'start_time': '00:00:33.320',
  'end_time': '00:00:34.240',
  'text': '传说曲'},
 {'index': '5',
  'start_time': '00:00:34.560',
  'end_time': '00:00:36.120',
  'text': '恶魔宠妃'},
 {'index': '6',
  'start_time': '00:00:38.240',
  'end_time': '00:00:39.160',
  'text': '都看完了吗'},
 {'index': '7',
  'start_time': '00:00:39.360',
  'end_time': '00:00:40.040',
  'text': '说说吧'},
 {'index': '8',
  'start_time': '00:00:40.320',
  'end_time': '00:00:41.400',
  'text': '大少客观啊'},
 {'index': '9',
  'start_time': '00:00:50.320',
  'end_time': '00:00:50.920',
  'text': '那个谁'},
 {'index': '10',
  'start_time': '00:00:51.520',
  'end_time': '00:00:52.200',
  'text': '

In [49]:
transcribe_video("https://www.youtube.com/watch?v=Q04CzDIZ5T4&t=64s", "zh")

[youtube] Extracting URL: https://www.youtube.com/watch?v=Q04CzDIZ5T4&t=64s
[youtube] Q04CzDIZ5T4: Downloading webpage


[youtube] Q04CzDIZ5T4: Downloading android vr player API JSON
[info] Q04CzDIZ5T4: Downloading 1 format(s): 140
[download] Destination: Q04CzDIZ5T4.m4a
[download] 100% of   93.89MiB in 00:00:09 at 9.78MiB/s     
[FixupM4a] Correcting container of "Q04CzDIZ5T4.m4a"
[ExtractAudio] Not converting audio Q04CzDIZ5T4.m4a; file is already in target format m4a
0


c:\Users\melis\.conda\envs\loqio-env\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|█████████▉| 608284/608314 [2:24:46<00:00, 70.03frames/s]  


[0.00s -> 8.00s] 世界本无天生的坦途
[8.00s -> 12.00s] 更无轻易可跨越的绝境
[12.00s -> 21.00s] 君不见世界乌及
[21.00s -> 23.00s] 心脏高原
[23.00s -> 27.00s] 万山之祖
[27.00s -> 29.00s] 帕米尔之巅
[30.00s -> 34.00s] 这般险峻高绝处
[34.00s -> 36.00s] 竟能有路
[36.00s -> 38.00s] 劈山脱道
[38.00s -> 40.00s] 破荒而来
[40.00s -> 45.00s] 这是当之无愧的天路
[49.00s -> 51.00s] 天路如经络
[51.00s -> 55.00s] 让高原秘境连通华夏大地
[55.00s -> 59.00s] 为边疆带来丰富变化
[59.00s -> 61.00s] 雪山
[62.00s -> 64.00s] 真的好漂亮 天哪
[64.00s -> 67.00s] 这里的风景要用眼睛看的
[68.00s -> 70.00s] 露出来了 那个尖
[70.00s -> 71.00s] 快拍
[71.00s -> 73.00s] 要许愿
[73.00s -> 75.00s] 好美
[78.00s -> 80.00s] 看我的眼泪
[80.00s -> 82.00s] 太漂亮了 天哪
[85.00s -> 87.00s] 好美啊
[87.00s -> 89.00s] 天路似寂凉
[89.00s -> 92.00s] 穿越生命禁区
[92.00s -> 94.00s] 为艰苦环境中的人们
[94.00s -> 98.00s] 撑起充满生机的苍穹
[98.00s -> 101.00s] 曾经我当过背负
[101.00s -> 103.00s] 走过这条路
[103.00s -> 105.00s] 翻过这条大山
[105.00s -> 107.00s] 我的天 这怎么翻过去啊
[107.00s -> 108.00s] 是正的上去
[108.00s -> 109.00s] 这里是扎帽公路
[109.00s -> 110.00s] 24K路段
[110.00s -> 112.00s] 这里有雪山 这里有冰川
[112.00s -> 113.00s] 隧道贯通之前
[113.00s ->

[{'index': '1',
  'start_time': '00:00:00.000',
  'end_time': '00:00:08.000',
  'text': '世界本无天生的坦途'},
 {'index': '2',
  'start_time': '00:00:08.000',
  'end_time': '00:00:12.000',
  'text': '更无轻易可跨越的绝境'},
 {'index': '3',
  'start_time': '00:00:12.000',
  'end_time': '00:00:21.000',
  'text': '君不见世界乌及'},
 {'index': '4',
  'start_time': '00:00:21.000',
  'end_time': '00:00:23.000',
  'text': '心脏高原'},
 {'index': '5',
  'start_time': '00:00:23.000',
  'end_time': '00:00:27.000',
  'text': '万山之祖'},
 {'index': '6',
  'start_time': '00:00:27.000',
  'end_time': '00:00:29.000',
  'text': '帕米尔之巅'},
 {'index': '7',
  'start_time': '00:00:30.000',
  'end_time': '00:00:34.000',
  'text': '这般险峻高绝处'},
 {'index': '8',
  'start_time': '00:00:34.000',
  'end_time': '00:00:36.000',
  'text': '竟能有路'},
 {'index': '9',
  'start_time': '00:00:36.000',
  'end_time': '00:00:38.000',
  'text': '劈山脱道'},
 {'index': '10',
  'start_time': '00:00:38.000',
  'end_time': '00:00:40.000',
  'text': '破荒而来'},
 {'index': 

In [43]:
transcribe_video("https://www.youtube.com/watch?v=oLHfnK9IVZs","zh")

[youtube] Extracting URL: https://www.youtube.com/watch?v=oLHfnK9IVZs
[youtube] oLHfnK9IVZs: Downloading webpage


[youtube] oLHfnK9IVZs: Downloading android vr player API JSON
[info] oLHfnK9IVZs: Downloading 1 format(s): 140
[download] oLHfnK9IVZs.m4a has already been downloaded
[download] 100% of   14.79MiB
[ExtractAudio] Not converting audio oLHfnK9IVZs.m4a; file is already in target format m4a
0


c:\Users\melis\.conda\envs\loqio-env\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 95912/95912 [07:23<00:00, 216.43frames/s]

[0.00s -> 27.00s] 早上好,我是佳佳。
[27.00s -> 34.00s] 我刚刚送了我的女儿去托儿所,
[34.00s -> 39.00s] 然后回家化了一个妆。
[39.00s -> 44.00s] 现在我准备做早餐。
[44.00s -> 50.00s] 今天我打算做我最喜欢的早餐。
[50.00s -> 54.00s] 需要的食材很简单。
[54.00s -> 62.00s] 牛油果、吐司、鸡蛋和黄油。
[62.00s -> 66.00s] 那我们先烤面包吧。
[69.00s -> 72.00s] 现在打一个鸡蛋。
[72.00s -> 85.00s] 现在开始煎蛋。
[85.00s -> 92.00s] 放一点黄油。
[92.00s -> 97.00s] 黄油融化以后,把蛋倒进去。
[97.00s -> 107.00s] 搅拌一下。
[107.00s -> 112.00s] 接下来就很简单了。
[112.00s -> 116.00s] 把牛油果抹上去。
[116.00s -> 130.00s] 哇,这个牛油果果核好大,可以用来水培。
[130.00s -> 144.00s] 然后再把鸡蛋放上去。
[144.00s -> 150.00s] 放一点黑胡椒。
[150.00s -> 156.00s] 放一点盐。
[156.00s -> 160.00s] 放一点黑胡椒。
[160.00s -> 166.00s] 我个人还喜欢放一点辣椒酱。
[166.00s -> 169.00s] 因为我喜欢吃辣。
[169.00s -> 173.00s] 放在盘子里。
[173.00s -> 176.00s] 现在做一杯咖啡。
[176.00s -> 183.00s] 这种咖啡壶大家都知道吧。
[183.00s -> 190.00s] 煮咖啡也很简单,只有几个步骤。
[190.00s -> 197.00s] 第一,倒水。
[197.00s -> 207.00s] 第二,放咖啡粉。
[207.00s -> 211.00s] 抖一下。
[211.00s -> 215.00s] 第三,放进去。
[215.00s -> 222.00s] 然后关上。
[222.00s -> 228.00s] 开火。
[228.00s -> 238.00s] 等个一两分钟。
[238.00

[{'index': '1',
  'start_time': '00:00:00.000',
  'end_time': '00:00:27.000',
  'text': '早上好,我是佳佳。'},
 {'index': '2',
  'start_time': '00:00:27.000',
  'end_time': '00:00:34.000',
  'text': '我刚刚送了我的女儿去托儿所,'},
 {'index': '3',
  'start_time': '00:00:34.000',
  'end_time': '00:00:39.000',
  'text': '然后回家化了一个妆。'},
 {'index': '4',
  'start_time': '00:00:39.000',
  'end_time': '00:00:44.000',
  'text': '现在我准备做早餐。'},
 {'index': '5',
  'start_time': '00:00:44.000',
  'end_time': '00:00:50.000',
  'text': '今天我打算做我最喜欢的早餐。'},
 {'index': '6',
  'start_time': '00:00:50.000',
  'end_time': '00:00:54.000',
  'text': '需要的食材很简单。'},
 {'index': '7',
  'start_time': '00:00:54.000',
  'end_time': '00:01:02.000',
  'text': '牛油果、吐司、鸡蛋和黄油。'},
 {'index': '8',
  'start_time': '00:01:02.000',
  'end_time': '00:01:06.000',
  'text': '那我们先烤面包吧。'},
 {'index': '9',
  'start_time': '00:01:09.000',
  'end_time': '00:01:12.000',
  'text': '现在打一个鸡蛋。'},
 {'index': '10',
  'start_time': '00:01:12.000',
  'end_time': '00:0